In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [3]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"beyzahiz","key":"80dc9015b76a77a6a15c99f1d215774b"}'}

In [4]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!kaggle datasets list

ref                                                              title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
nalisha/job-salary-prediction-dataset                            Job Salary Prediction Dataset                          3144815  2026-03-16 19:54:33.843000           9158        220                1  
ssssws/chocolate-sales-dataset-2023-2024                         Chocolate Sales Dataset 2023 - 2024                   24420255  2026-03-07 04:58:02.387000          12790        205                1  
sumeakash/impact-of-social-media-on-health                       Impact of Social Media on Health                         18598  2026-04-05 06:53:55.953000           1665         44        0.94117

In [6]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
!unzip brain-tumor-mri-dataset.zip

Görüntülenen çıkış son 5000 satıra kısaltıldı.
  inflating: Training/glioma/Tr-gl_279.jpg  
  inflating: Training/glioma/Tr-gl_28.jpg  
  inflating: Training/glioma/Tr-gl_280.jpg  
  inflating: Training/glioma/Tr-gl_281.jpg  
  inflating: Training/glioma/Tr-gl_282.jpg  
  inflating: Training/glioma/Tr-gl_283.jpg  
  inflating: Training/glioma/Tr-gl_284.jpg  
  inflating: Training/glioma/Tr-gl_285.jpg  
  inflating: Training/glioma/Tr-gl_286.jpg  
  inflating: Training/glioma/Tr-gl_287.jpg  
  inflating: Training/glioma/Tr-gl_288.jpg  
  inflating: Training/glioma/Tr-gl_289.jpg  
  inflating: Training/glioma/Tr-gl_29.jpg  
  inflating: Training/glioma/Tr-gl_290.jpg  
  inflating: Training/glioma/Tr-gl_291.jpg  
  inflating: Training/glioma/Tr-gl_292.jpg  
  inflating: Training/glioma/Tr-gl_293.jpg  
  inflating: Training/glioma/Tr-gl_294.jpg  
  inflating: Training/glioma/Tr-gl_295.jpg  
  inflating: Training/glioma/Tr-gl_296.jpg  
  inflating: Training/glioma/Tr-gl_297.jpg  
  inflatin

In [7]:
!ls

brain-tumor-mri-dataset.zip  kaggle.json  sample_data  Testing	Training


In [8]:
!ls Training

glioma	meningioma  notumor  pituitary


In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_dir = "Training"
test_dir = "Testing"

train_datagen = ImageDataGenerator(
    rescale=1./255, #görseller 0-255 iken 0-1.. yapar
    rotation_range=10, #Görselleri ±10 derece döndürür
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory( #bu fonksiyon klasörleri okur, görselleri yükler, label üretir
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)


Found 5600 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.


In [10]:
print(train_generator.class_indices)

{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [11]:
#EfficentNet Modeli

from tensorflow.keras.applications import EfficientNetB0
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

for layer in base_model.layers:
  layer.trainable = False

from tensorflow.keras import layers,models

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation ='relu')(x)
output = layers.Dense(4, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,214,055 (16.08 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [12]:
history = model.fit(
    train_generator,
    validation_data = test_generator, #her epoch sonunda test verisiyle kendini kontrol eder, overfitting için önemli
    epochs=5,
)

model.save("mri_model.h5")

Epoch 1/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 108s 460ms/step - accuracy: 0.2489 - loss: 1.4007 - val_accuracy: 0.2500 - val_loss: 1.3854
Epoch 2/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 70s 397ms/step - accuracy: 0.2587 - loss: 1.3872 - val_accuracy: 0.2500 - val_loss: 1.3870
Epoch 3/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 71s 406ms/step - accuracy: 0.2586 - loss: 1.3862 - val_accuracy: 0.2537 - val_loss: 1.3858
Epoch 4/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 71s 405ms/step - accuracy: 0.2546 - loss: 1.3854 - val_accuracy: 0.3275 - val_loss: 1.3842
Epoch 5/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 71s 404ms/step - accuracy: 0.2704 - loss: 1.3840 - val_accuracy: 0.2706 - val_loss: 1.3818


In [13]:
#Fine-tuning

for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=5
)

model.save("mri_model.keras")

Epoch 1/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 104s 449ms/step - accuracy: 0.3187 - loss: 1.3504 - val_accuracy: 0.2519 - val_loss: 1.3859
Epoch 2/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 72s 409ms/step - accuracy: 0.3871 - loss: 1.2603 - val_accuracy: 0.3056 - val_loss: 1.3428
Epoch 3/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 71s 404ms/step - accuracy: 0.4121 - loss: 1.2244 - val_accuracy: 0.3800 - val_loss: 1.2665
Epoch 4/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 72s 409ms/step - accuracy: 0.4361 - loss: 1.1943 - val_accuracy: 0.4087 - val_loss: 1.3308
Epoch 5/5
175/175 ━━━━━━━━━━━━━━━━━━━━ 70s 401ms/step - accuracy: 0.4616 - loss: 1.1631 - val_accuracy: 0.3487 - val_loss: 1.3088
